# NB02 — Biome Classification & Coverage-by-Biome Aggregation

**Environment:** BERDL JupyterHub (Spark).

**Purpose:**
1. Classify every genome into one of 17 biome labels (host_gut, soil, marine, etc.) from `gtdb_metadata.ncbi_isolation_source` keyword matching.
2. Assign each species a majority-vote biome across its genomes.
3. Join per-cluster Pfam tier (from NB01) to species biome and pangenome core/accessory status.
4. Aggregate to biome × pfam_tier × is_core cell matrix + per-biome summary + top uncovered Pfams per biome.

**Why keyword classifier, not the `plant_microbiome_ecotypes` `genome_environment.csv`:** that CSV lives in a different user's home directory and is not on this cluster's filesystem. `gtdb_metadata.ncbi_isolation_source` covers 78.7% of genomes and is available in-BERDL.

**Full script:** `scripts/01_extract_and_stratify.py` runs the entire NB01+NB02 pipeline and writes all local CSVs into `data/`.

**Outputs (in `data/`):**
- `biome_summary.csv` (18 biomes × 11 metrics)
- `biome_pfam_matrix.csv` (145 rows, biome × tier × is_core)
- `biome_top_uncovered.csv` (340 rows, top-20 uncovered Pfam per biome)
- `genome_biome.csv` (226K genomes with biome labels)
- `species_biome.csv` (26.5K species with majority-vote biome)

In [ ]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql.functions import (col, when, lit, count, countDistinct,
    sum as spark_sum, row_number, regexp_replace, lower, desc)
from pyspark.sql.window import Window

spark = get_spark_session()

## Biome classifier (keyword rules on isolation source)

In [ ]:
biome_rules = [
    ('host_gut',           r'\b(gut|feces|stool|faecal|fecal|cecum|caecum|cecal|caecal|rumen|colon|rectum|rectal|gastrointestinal|gi tract|intestin|colonic|coprolite|mucosa)\b'),
    ('host_respiratory',   r'\b(lung|sputum|nasopharyn|oropharyn|respiratory|oral|saliva|tongue|throat|bronch|tonsil|dental|plaque|tooth|teeth|nasal)\b'),
    ('host_urogenital',    r'\b(urin|urethra|vagina|cervi|urogenital|bladder|prostate|semen|penile)\b'),
    ('host_blood_tissue',  r'\b(blood|serum|plasma|csf|cerebrospinal|wound|abscess|pus|tissue|biopsy|liver|kidney|spleen|lymph|joint|synovial|bone marrow)\b'),
    ('host_skin',          r'\b(skin|dermal|epidermis|scalp|sebaceous|axilla|forearm|dermat)\b'),
    ('host_other',         r'\b(human|patient|clinical|hospital|host|homo sapiens|infant|neonate|elderly)\b'),
    ('plant_associated',   r'\b(plant|phyllosphere|endosphere|endophyte|rhizosphere|rhizoplane|root|leaf|leaves|shoot|stem|seed|flower|fruit|nodul|arabidopsis|maize|wheat|rice|soybean|tomato|potato|barley|sorghum)\b'),
    ('soil',               r'\b(soil|permafrost|subsoil|topsoil|paddy|farmland|arable|dryland|grassland|tundra|desert|rhizosphere soil)\b'),
    ('sediment',           r'\b(sediment|mud|slime|silt|riverbed|lakebed|seabed|estuar)\b'),
    ('marine',             r'\b(marine|seawater|sea water|ocean|coastal|saline water|halocline|reef|coral|sponge|kelp|algae|deep-sea|planktonic|zooplankton|hypoxic seawater)\b'),
    ('freshwater',         r'\b(lake water|lake|river|stream|freshwater|pond|reservoir|groundwater|spring water|aquifer|surface water|water sample|water \(|wetland)\b'),
    ('subsurface_extreme', r'\b(subsurface|borehole|deep subsurface|deep-sea vent|hydrothermal|hot spring|geyser|acid mine|acidic|thermophil|hyperthermophil|psychrophil|halophil|cave|mine drainage|underground)\b'),
    ('built_environment',  r'\b(sewage|wastewater|activated sludge|bioreactor|biofilm|hospital surface|air sample|indoor|shower|toilet|cooling tower|drink water|drinking water)\b'),
    ('food_industrial',    r'\b(food|dairy|milk|cheese|yogurt|fermented|sourdough|kimchi|sauerkraut|beer|wine|kombucha|sausage|meat|fish product|kefir|salami|cured)\b'),
    ('agricultural_animal',r'\b(cattle|bovine|chicken|pig|swine|poultry|sheep|goat|horse|equine|canine|dog|feline|cat|silage|feedlot|dairy cow|calf|piglet)\b'),
    ('insect_invertebrate',r'\b(insect|bee|honeybee|ant|termite|larva|larvae|beetle|wasp|butterfly|nematode|worm|drosophila|caterpillar|gut of|midgut|hindgut)\b'),
]

gt = (spark.table('kbase_ke_pangenome.gtdb_metadata')
    .select(col('accession').alias('genome_id'), 'ncbi_isolation_source')
    .filter("ncbi_isolation_source IS NOT NULL AND ncbi_isolation_source NOT IN ('none','not known','not provided','not applicable','missing','unknown','Unknown','N/A','NA')"))

src = lower(col('ncbi_isolation_source'))
case_expr = None
for label, pattern in biome_rules:
    cond = src.rlike(pattern)
    case_expr = when(cond, lit(label)) if case_expr is None else case_expr.when(cond, lit(label))
case_expr = case_expr.otherwise(lit('other'))

genome_biome = gt.withColumn('biome', case_expr)
genome_biome.groupBy('biome').agg(count('*').alias('n')).orderBy(desc('n')).show(20)

## Species majority-vote biome

In [ ]:
genome = spark.table('kbase_ke_pangenome.genome').select('genome_id', 'gtdb_species_clade_id')
species_counts = (genome.join(genome_biome, on='genome_id', how='inner')
    .groupBy('gtdb_species_clade_id', 'biome').agg(count('*').alias('n_genomes')))
w = Window.partitionBy('gtdb_species_clade_id').orderBy(desc('n_genomes'))
species_biome = (species_counts.withColumn('rk', row_number().over(w))
    .filter(col('rk') == 1).drop('rk'))

## Full join & aggregate

See `scripts/01_extract_and_stratify.py` for the full pipeline that produces `biome_summary.csv`, `biome_pfam_matrix.csv`, `biome_top_uncovered.csv`. The heavy Spark work is: recompute per-cluster tier (from NB01), join to gene_cluster (for species + is_core), join to species_biome, then aggregate.